# News Topic Classification on AG News
## Sequential (BiLSTM) vs. Transformer Encoder — both implemented from scratch

**Case study — Transformer Modeling and Sequence Learning (Topic 2)**

**Research question.** *How do recurrent networks and Transformers compare at classifying short news
texts?*

**Dataset.** AG News (Kaggle: `amananandrai/ag-news-classification-dataset`) — 120,000 training and
7,600 test articles over four topics: World, Sports, Business, Sci/Tech. Following the brief, a
**controlled 60,000-record** stratified subset is used for training/validation, while the **full
official 7,600-record test set** is held back for the final comparison.

**What "from scratch" means here.** PyTorch is used for tensors, autograd and the optimiser.
`nn.Linear`, `nn.Embedding` and `nn.Dropout` are used purely as parameter containers. Every
sequence-modelling component is written out by hand:

| Component | Written by hand in this notebook |
|---|---|
| LSTM cell (input/forget/cell/output gates) | `LSTMCellScratch` |
| Recurrence over time + bidirectionality | `LSTMLayerScratch` |
| Scaled dot-product attention | `MultiHeadSelfAttentionScratch` |
| Multi-head split / merge projections | `MultiHeadSelfAttentionScratch` |
| Sinusoidal positional encoding | `sinusoidal_positional_encoding` |
| Layer normalisation | `LayerNormScratch` |
| Position-wise feed-forward + residual block | `EncoderBlockScratch` |

**Not used anywhere in the models:** `nn.LSTM`, `nn.GRU`, `nn.RNN`, `nn.MultiheadAttention`,
`nn.Transformer*`, `nn.LayerNorm`, `F.scaled_dot_product_attention`. `nn.LSTM` and
`F.layer_norm` appear **once each, inside Section 5**, used as a numerical reference to prove the
hand-written layers are correct.

**Structure.** 1 Data → 2 Controlled experiment setup → 3 BiLSTM from scratch →
4 Transformer from scratch → 5 Correctness tests → 6 Shared training harness →
7 Comparative evaluation → 8 Conclusion.

## 0. Configuration and reproducibility

`SMOKE_TEST = True` shrinks the whole notebook to a ~1 minute end-to-end run (2,000 records,
1 epoch) — useful for checking the notebook executes before committing to the full run.
Set it to `False` for the real experiment.

In [ ]:
SMOKE_TEST = False   # True -> 2,000 records / 1 epoch, for a fast end-to-end check

# ---- experiment configuration -------------------------------------------------
SEED          = 42
SUBSET_SIZE   = 60_000     # controlled subset drawn from the 120k training file
VAL_FRACTION  = 0.10       # -> 54,000 train / 6,000 validation
MAX_LEN       = 64         # tokens per document (truncate / pad)
MIN_FREQ      = 2          # vocabulary cutoff, computed on the training split only

# ---- shared training budget (identical for both models) -----------------------
BATCH_SIZE    = 128
MAX_EPOCHS    = 8
PATIENCE      = 2          # early stopping on validation macro-F1
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.01

# ---- per-architecture learning rates (tuned on validation only) ---------------
LR_BILSTM      = 1e-3
LR_TRANSFORMER = 3e-4

# ---- architecture sizes -------------------------------------------------------
EMB_DIM        = 128       # shared embedding width, so both models see the same input capacity
LSTM_HIDDEN    = 128
LSTM_DROPOUT   = 0.30
D_MODEL        = 128
N_HEADS        = 4
N_LAYERS       = 2
D_FF           = 256
TRF_DROPOUT    = 0.10

if SMOKE_TEST:
    SUBSET_SIZE, MAX_EPOCHS, PATIENCE = 2_000, 1, 1

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
N_CLASSES = len(LABELS)

DATA_DIR = "data/agnews"
ART_DIR  = "artifacts"

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}  |  subset={SUBSET_SIZE:,}  epochs={MAX_EPOCHS}")

In [ ]:
import json, math, os, random, re, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
Path(ART_DIR).mkdir(exist_ok=True)


def set_seed(seed: int = SEED) -> None:
    # Seed every source of randomness we touch, so the whole notebook is reproducible.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE} | threads: {torch.get_num_threads()}")

In [ ]:
# ---- plotting style ------------------------------------------------------------
# Categorical palette in fixed slot order (blue, orange, aqua, yellow). Colours are
# assigned by entity (model / class) and never re-assigned when a chart's series change.
SERIES   = {"BiLSTM": "#2a78d6", "Transformer": "#eb6834"}
CLASS_C  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK      = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8880"}
GRID     = "#e3e2de"
SURFACE  = "#fcfcfb"
# Single-hue sequential ramp (light -> dark blue) for magnitude, e.g. confusion matrices.
BLUE_SEQ = LinearSegmentedColormap.from_list(
    "blue_seq", ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK["secondary"],
    "axes.titlecolor": INK["primary"], "axes.titlesize": 12, "axes.titleweight": "600",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.bottom": True, "ytick.left": True, "xtick.direction": "out",
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.alpha": 1.0,
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK["secondary"],
    "lines.linewidth": 2.0, "lines.markersize": 8,
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})


def recede(ax, axis="y"):
    # Grid and axes stay recessive: one faint set of gridlines behind the data, nothing else.
    ax.grid(axis=axis, zorder=0)
    ax.set_axisbelow(True)
    return ax


def save_fig(fig, name):
    path = Path(ART_DIR) / name
    fig.savefig(path)
    print(f"saved -> {path}")

## 1. Data acquisition

The dataset is pulled through the Kaggle API so the source is reproducible and citable. The
newer Kaggle token format (`KGAT_...`) is read from `~/.kaggle/access_token` or the
`KAGGLE_API_TOKEN` environment variable.

The CSVs are downloaded into their own directory and only ever read with `pandas.read_csv` —
nothing from the download directory is executed.

In [ ]:
def ensure_agnews(data_dir: str = DATA_DIR) -> tuple[Path, Path]:
    # Download AG News from Kaggle unless it is already on disk.
    train_csv, test_csv = Path(data_dir) / "train.csv", Path(data_dir) / "test.csv"
    if train_csv.exists() and test_csv.exists():
        print(f"Found cached dataset in {data_dir}/ - skipping download.")
        return train_csv, test_csv

    token_file = Path.home() / ".kaggle" / "access_token"
    legacy     = Path.home() / ".kaggle" / "kaggle.json"
    if not (token_file.exists() or legacy.exists() or os.environ.get("KAGGLE_API_TOKEN")):
        raise RuntimeError(
            "No Kaggle credentials found. Do one of:\n"
            "  mkdir -p ~/.kaggle && echo KGAT_xxx > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
            "  export KAGGLE_API_TOKEN=KGAT_xxx\n"
            "  (legacy) place username/key JSON at ~/.kaggle/kaggle.json with chmod 600\n"
            "Get a token from https://www.kaggle.com/settings")

    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    api.dataset_download_files("amananandrai/ag-news-classification-dataset",
                               path=data_dir, unzip=True, quiet=True)
    print(f"Downloaded AG News to {data_dir}/")
    return train_csv, test_csv


TRAIN_CSV, TEST_CSV = ensure_agnews()

In [ ]:
# The raw CSVs carry literal escape artefacts from the original scrape, e.g.
# "dwindling\band of ultra-cynics" and "\n" inside Description. Left in place, the
# tokeniser would weld two real words into one junk type, so they are stripped first.
_ESCAPES = re.compile(r"\\[a-z]")


def load_split(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"Class Index": "class_index"})
    title = df["Title"].fillna("").astype(str)
    desc  = df["Description"].fillna("").astype(str)
    text  = (title.str.strip() + ". " + desc.str.strip())
    text  = text.str.replace(_ESCAPES, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    out = pd.DataFrame({"text": text, "label": df["class_index"].astype(int) - 1})
    before = len(out)
    out = out[(out["text"].str.len() > 0) & out["label"].between(0, N_CLASSES - 1)].reset_index(drop=True)
    print(f"{csv_path.name}: {before:,} rows -> {len(out):,} usable ({before - len(out)} dropped)")
    return out


train_full = load_split(TRAIN_CSV)
test_df    = load_split(TEST_CSV)

print("\nExample records:")
for i in [0, 1, 2]:
    print(f"  [{LABELS[train_full.label[i]]:<9}] {train_full.text[i][:95]}...")

## 2. The controlled experiment

Three things make this a *controlled* comparison rather than two unrelated training runs:

1. **One stratified 60,000-record subset** (15,000 per class) drawn from the training file with a
   fixed seed, split 54,000 / 6,000 into train and validation.
2. **One shared preprocessing pipeline** — the same tokeniser, the same vocabulary (built on the
   training split only, so no leakage), the same `MAX_LEN`, and the same integer tensors are fed to
   both models.
3. **One shared training harness** (Section 6) with an identical budget: same batch size, same
   maximum epochs, same optimiser family, same early-stopping rule, same seed.

The only thing that differs between the two runs is **the architecture** (and its learning rate,
discussed in Section 6).

In [ ]:
set_seed()

# Stratified subset: exactly SUBSET_SIZE // 4 records per class.
per_class = SUBSET_SIZE // N_CLASSES
class_samples = [train_full[train_full.label == c].sample(
                     n=min(per_class, int((train_full.label == c).sum())), random_state=SEED)
                 for c in range(N_CLASSES)]
subset = (pd.concat(class_samples)
            .sample(frac=1.0, random_state=SEED)      # shuffle so splits are not class-ordered
            .reset_index(drop=True))

train_df, val_df = train_test_split(
    subset, test_size=VAL_FRACTION, random_state=SEED, stratify=subset["label"])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

dist = pd.DataFrame({
    "Train": train_df.label.value_counts().sort_index(),
    "Validation": val_df.label.value_counts().sort_index(),
    "Test (official)": test_df.label.value_counts().sort_index(),
}).rename(index=dict(enumerate(LABELS)))
dist.loc["TOTAL"] = dist.sum()
print("Class distribution across splits\n")
print(dist.to_string())
print(f"\nControlled subset: {len(subset):,} records -> {len(train_df):,} train / {len(val_df):,} val")
print(f"Held-out official test set: {len(test_df):,} records")

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9']+")
PAD_ID, UNK_ID = 0, 1


def tokenize(text: str) -> list[str]:
    # Lowercase + keep alphanumeric/apostrophe runs. Deliberately simple: both models get the
    # same tokens, so tokeniser sophistication cannot favour one architecture over the other.
    return TOKEN_RE.findall(text.lower())


# Vocabulary from the TRAINING split only.
counts = Counter()
train_tokens = [tokenize(t) for t in train_df.text]
for toks in train_tokens:
    counts.update(toks)

itos = ["<pad>", "<unk>"] + [w for w, c in counts.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)

print(f"Token types seen in training split : {len(counts):,}")
print(f"Vocabulary (min_freq >= {MIN_FREQ})       : {VOCAB_SIZE:,}")
print(f"Most common: {[w for w, _ in counts.most_common(12)]}")

In [ ]:
lengths = np.array([len(t) for t in train_tokens])
coverage = (lengths <= MAX_LEN).mean()

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.hist(lengths, bins=np.arange(0, 121, 2), color=SERIES["BiLSTM"], edgecolor=SURFACE, linewidth=0.6, zorder=2)
ax.axvline(MAX_LEN, color=INK["primary"], linestyle="--", linewidth=1.5, zorder=3)
ax.annotate(f"MAX_LEN = {MAX_LEN}\n{coverage:.1%} of documents fit untruncated",
            xy=(MAX_LEN, ax.get_ylim()[1] * 0.92), xytext=(MAX_LEN + 5, ax.get_ylim()[1] * 0.92),
            color=INK["primary"], fontsize=9, va="top")
ax.set(title="Document length in tokens (training split)", xlabel="tokens", ylabel="documents", xlim=(0, 120))
recede(ax)
fig.tight_layout(); save_fig(fig, "fig0_token_lengths.png"); plt.show()

print(f"median={np.median(lengths):.0f}  mean={lengths.mean():.1f}  p95={np.percentile(lengths, 95):.0f}  max={lengths.max()}")
print(f"Coverage at MAX_LEN={MAX_LEN}: {coverage:.2%} of training documents are not truncated.")

In [ ]:
def encode(texts, max_len: int = MAX_LEN):
    # -> (ids[N, max_len] int64, lengths[N] int64). Right-padded with PAD_ID.
    n = len(texts)
    ids  = np.zeros((n, max_len), dtype=np.int64)
    lens = np.zeros(n, dtype=np.int64)
    for i, text in enumerate(texts):
        toks = tokenize(text)[:max_len]
        if not toks:                       # keep every row non-empty so masks never sum to 0
            toks = ["<unk>"]
        ids[i, :len(toks)] = [stoi.get(t, UNK_ID) for t in toks]
        lens[i] = len(toks)
    return torch.from_numpy(ids), torch.from_numpy(lens)


def make_dataset(df):
    ids, lens = encode(df.text.tolist())
    return TensorDataset(ids, lens, torch.tensor(df.label.values, dtype=torch.long))


t0 = time.time()
train_ds, val_ds, test_ds = make_dataset(train_df), make_dataset(val_df), make_dataset(test_df)
print(f"Encoded {len(train_ds) + len(val_ds) + len(test_ds):,} documents in {time.time() - t0:.1f}s")

g = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g, drop_last=False)
val_dl   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False)

xb, lb, yb = next(iter(train_dl))
print(f"batch ids {tuple(xb.shape)} | lengths {tuple(lb.shape)} | labels {tuple(yb.shape)}")
print(f"first row decoded: {' '.join(itos[i] for i in xb[0][:lb[0]])[:110]}...")